# Intercropping PDF evidence pack

This notebook reads the **completed** parentage-likelihood analysis. It does not download imagery, rerun TESSERA, refit a model, or rebuild the full field gallery.

Run all cells, then send the numbered outputs back in the chat. Output 1 is copyable text. Outputs 2-7 are the small set of figures needed for the team PDF. Representative examples are selected at `w3`, the last in-contract window, using cohort-median evidence rather than the strongest-looking result. Every field dashboard still shows `w1` through `w4`; `w4` remains an OOD sensitivity window.

In [ ]:
import json
import os

from IPython import get_ipython
from IPython.display import Image, Markdown, display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

from plain_tessera_incremental.notebooks._pdf_evidence_pack import (
    DEFAULT_ANALYSIS_ROOT,
    field_figure_path,
    find_completed_analysis_dir,
    load_evidence_pack,
    mixture_outcomes,
    overview_figure_path,
    plot_mixture_outcomes,
    presentation_facts,
    select_representative_fields,
)

explicit_dir = os.environ.get("TESSERA_DNA_ANALYSIS_DIR")
analysis_dir = find_completed_analysis_dir(
    DEFAULT_ANALYSIS_ROOT,
    explicit=explicit_dir,
)
pack = load_evidence_pack(analysis_dir)
examples = select_representative_fields(pack)
print("Completed analysis:", analysis_dir)
print("Representative dashboards selected:", len(examples))

## Output 1 - copy all text

This is the complete numerical fact sheet for the PDF: provenance, denominators, four windows, validation, mixture outcomes, monocrop controls, and every call status.

In [ ]:
print("PDF_FACTS_BEGIN")
print(json.dumps(presentation_facts(pack), indent=2, allow_nan=False))
print("PDF_FACTS_END")

## Output 2 - copy this cohort overview

This is the all-field view. It preserves denominators and shows evidence relative to pure-crop controls, longitudinal mixture balance, and call statuses.

In [ ]:
display(Image(filename=str(overview_figure_path(pack)), width=1200, embed=True))

## Output 3 - copy this incremental-results figure

Solid lines show supported intercropping calls, dashed lines show fields rejected as out of model, and the second panel follows the field-level parent signature through the cumulative windows.

In [ ]:
outcome_figure = plot_mixture_outcomes(
    mixture_outcomes(pack),
    windows=pack.manifest["windows"],
)
display(outcome_figure)
plt.close(outcome_figure)

## Selection audit - include this text with the figures

This table records exactly why each example was chosen. It prevents accidental cherry-picking.

In [ ]:
print("PDF_EXAMPLE_SELECTION_BEGIN")
print(examples.to_csv(index=False))
print("PDF_EXAMPLE_SELECTION_END")

def show_example(role):
    selected = examples[examples["role"].astype(str).eq(role)]
    if selected.empty:
        print(f"No available example for: {role}")
        return
    row = selected.iloc[0]
    display(Markdown(f"**{role}** - `{row['field_uid']}` - {row['selection_basis']}"))
    display(
        Image(
            filename=str(field_figure_path(pack, str(row["field_uid"]))),
            width=1350,
            embed=True,
        )
    )

## Output 4 - representative Bean and Maize field

In [ ]:
show_example("representative Bean and Maize")

## Output 5 - representative Irish Potato and Maize field

In [ ]:
show_example("representative Irish Potato and Maize")

## Output 6 - monocrop negative control

In [ ]:
show_example("monocrop negative control")

## Output 7 - model guardrail example

This example demonstrates that normalized probabilities are not accepted when the field lies outside the held-out reference distribution.

In [ ]:
show_example("model guardrail example")

## What to send

Send Output 1 as text, the selection-audit text, and Outputs 2-7 as copied notebook images. If Output 7 says that no guardrail example is available, send that message instead. No Parquet files, embeddings, or gallery archive are needed.